# Composite ranking: significance + |delta_psi_treatment|

Two-axis composite per gene, per lineage:

$$\text{composite\_score\_abs} = 0.50 \cdot \text{scaled}[-\log_{10}(\text{padj\_treatment})] + 0.50 \cdot \text{rank\_pct}(|\Delta\psi_{\text{treatment}}|)$$

$$\text{composite\_score\_signed} = \mathrm{sign}(\Delta\psi_{\text{treatment}}) \cdot \text{composite\_score\_abs}$$

Inputs: the per-lineage `metrics_per_gene.csv` produced by `05_260512-calculate_metrics.ipynb`. The test column is `padj_treatment` from the primary main-effect screen.

Two axes, without entropy. Entropy (`delta_entropy`) is reported in `metrics_per_gene.csv` but not used here: the primary endpoint for the treatment contrast is the centroid shift, captured by `delta_psi_treatment`. Entropy is kept in the metrics table for visualisation and as a secondary, hypothesis-generating signal.

No re-FDR step. DRIMSeq already BH-adjusts `pvalue_treatment` over the post-filter gene set inside the R script. Re-correcting here would double-adjust.


## Locations

Paths are relative to the data directory given by the environment variable `M26_DATA_ROOT` (see the README); figure panels are written to `M26_RESULTS_ROOT`.

In [ ]:
# Locations (see README): M26_DATA_ROOT is required; M26_RESULTS_ROOT defaults to <repo>/results
import os
from pathlib import Path

REPO_ROOT = os.environ.get("M26_REPO_ROOT") or str(
    next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "environment" / "apa-python.yml").is_file())
)
os.environ["M26_REPO_ROOT"] = REPO_ROOT
DATA_ROOT = os.environ["M26_DATA_ROOT"]
RESULTS_ROOT = os.environ.get("M26_RESULTS_ROOT", f"{REPO_ROOT}/results")
# Run in the directory this notebook originally ran in, so that relative paths resolve as they did
WORK_DIR = Path(DATA_ROOT, "notebooks/3UTR/organoids_knoblich/UTR_treatment")
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)

In [ ]:
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

warnings.filterwarnings("ignore")
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 300, "font.size": 7,
    "font.family": "sans-serif", "font.sans-serif": ["Arial","Helvetica","DejaVu Sans"],
    "axes.titlesize": 8, "axes.labelsize": 7,
    "axes.spines.top": False, "axes.spines.right": False,
    "pdf.fonttype": 42, "ps.fonttype": 42, "axes.linewidth": 0.8,
})
print("Setup complete.")

In [ ]:
LINEAGES = ["inhibitory", "excitatory"]
W_PADJ, W_DPSI = 0.50, 0.50

BASEPATH = Path(f"{DATA_ROOT}/3utr/organoids/knoblich")
UTR_DIR = BASEPATH / "utrcalling"
RESULTS_ROOT = UTR_DIR / "dirmult_treatment_results"

def metrics_csv(lineage):
    return RESULTS_ROOT / lineage / "calculate_metrics" / "metrics_per_gene.csv"

def out_dir(lineage):
    p = RESULTS_ROOT / lineage / "composite_ranking"; p.mkdir(parents=True, exist_ok=True); return p

for ln in LINEAGES:
    assert metrics_csv(ln).exists(), f"missing {metrics_csv(ln)}"
    print(f"  {ln:12s}  in={metrics_csv(ln)}\n               out={out_dir(ln)}")

## Build the composite per lineage

In [ ]:
def build_composite(lineage):
    df = pd.read_csv(metrics_csv(lineage))
    n_pre = len(df)
    df = df[df["delta_psi_treatment"].notna() & df["padj_treatment"].notna()].copy()
    print(f"[{lineage}] {n_pre:,} genes; after non-NaN padj+dpsi: {len(df):,}")

    neglog = -np.log10(df["padj_treatment"].clip(lower=1e-300))
    rng = neglog.max() - neglog.min()
    df["scaled_neglog10_padj_treatment"] = (neglog - neglog.min()) / rng if rng > 0 else 0.0
    df["abs_delta_psi"] = df["delta_psi_treatment"].abs()
    df["rank_abs_delta_psi"] = df["abs_delta_psi"].rank(pct=True)
    df["composite_score_abs"] = W_PADJ * df["scaled_neglog10_padj_treatment"] \
                              + W_DPSI * df["rank_abs_delta_psi"]
    df["composite_score_signed"] = np.sign(df["delta_psi_treatment"]) * df["composite_score_abs"]
    df = df.sort_values("composite_score_abs", ascending=False).reset_index(drop=True)

    out_csv = out_dir(lineage) / f"composite_ranking_{lineage}.csv"
    df.to_csv(out_csv, index=False)
    print(f"  wrote {df.shape}  -> {out_csv}")
    return df

composites = {ln: build_composite(ln) for ln in LINEAGES}
composites["inhibitory"].head(10)

## Per-lineage distributions of the score components

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for row, ln in enumerate(LINEAGES):
    df = composites[ln]
    axes[row,0].hist(df["scaled_neglog10_padj_treatment"], bins=60, color="#c0392b", edgecolor="black", linewidth=0.2)
    axes[row,0].set_title(f"{ln}: scaled_neglog10_padj_treatment"); axes[row,0].set_xlabel("score component"); axes[row,0].set_ylabel("genes")
    axes[row,1].hist(df["rank_abs_delta_psi"], bins=60, color="#2980b9", edgecolor="black", linewidth=0.2)
    axes[row,1].set_title(f"{ln}: rank_pct(|delta_psi|)"); axes[row,1].set_xlabel("score component")
    axes[row,2].hist(df["composite_score_abs"], bins=60, color="#16a085", edgecolor="black", linewidth=0.2)
    axes[row,2].set_title(f"{ln}: composite_score_abs"); axes[row,2].set_xlabel("composite_score_abs")
    axes[row,3].hist(df["composite_score_signed"], bins=60, color="grey", edgecolor="black", linewidth=0.2)
    axes[row,3].axvline(0, color="black", lw=0.6, ls="--")
    axes[row,3].set_title(f"{ln}: composite_score_signed"); axes[row,3].set_xlabel("composite_score_signed")
plt.tight_layout(); plt.show()

## Cross-lineage concordance

In [ ]:
inh = composites["inhibitory"].set_index("gene_id")
exc = composites["excitatory"].set_index("gene_id")
shared = sorted(set(inh.index) & set(exc.index))
print(f"Genes shared post-filter across both lineages: {len(shared):,}")
print(f"  inhibitory-only: {len(set(inh.index) - set(exc.index)):,}")
print(f"  excitatory-only: {len(set(exc.index) - set(inh.index)):,}")

inh_s = inh.loc[shared, ["composite_score_abs","composite_score_signed","delta_psi_treatment","delta_entropy","padj_treatment"]]
exc_s = exc.loc[shared, ["composite_score_abs","composite_score_signed","delta_psi_treatment","delta_entropy","padj_treatment"]]

print("\nSpearman correlations (shared genes):")
for col in ["composite_score_abs","composite_score_signed","delta_psi_treatment","delta_entropy"]:
    valid = inh_s[col].notna() & exc_s[col].notna()
    if valid.sum() < 5:
        print(f"  {col:30s} n={valid.sum()}  (too few)"); continue
    rho, p = spearmanr(inh_s.loc[valid, col], exc_s.loc[valid, col])
    print(f"  {col:30s} rho={rho:+.3f}  p={p:.2e}  n={valid.sum():,}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, col, lo, hi in [
    (axes[0], "composite_score_abs", 0, 1),
    (axes[1], "delta_psi_treatment", -1, 1),
    (axes[2], "delta_entropy", -1, 1),
]:
    valid = inh_s[col].notna() & exc_s[col].notna()
    x = inh_s.loc[valid, col].values; y = exc_s.loc[valid, col].values
    ax.scatter(x, y, s=3, alpha=0.3, c="#555555", rasterized=True)
    ax.plot([lo, hi],[lo, hi], color="firebrick", ls="--", lw=0.7)
    rho, _ = spearmanr(x, y)
    ax.set_xlabel(f"inhibitory {col}"); ax.set_ylabel(f"excitatory {col}")
    ax.set_title(f"{col}\nSpearman={rho:.3f} (n={valid.sum():,})")
    if col == "composite_score_abs":
        ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
plt.tight_layout(); plt.show()

## Top 25 per lineage by composite_score_abs

In [ ]:
for ln in LINEAGES:
    df = composites[ln].head(25)
    print(f"\n=== TOP 25 by composite_score_abs -- {ln} ===")
    cols = ["gene_id","n_features_post_filter","padj_treatment","delta_psi_treatment",
            "rank_abs_delta_psi","scaled_neglog10_padj_treatment",
            "composite_score_abs","composite_score_signed","pattern",
            "delta_entropy","entropy_padj_bh"]
    avail = [c for c in cols if c in df.columns]
    with pd.option_context("display.float_format", lambda v: f"{v:.3e}" if abs(v) < 1e-2 else f"{v:.4f}"):
        print(df[avail].to_string(index=False))

## Summary

In [ ]:
for ln in LINEAGES:
    df = composites[ln]
    print(f"\n[{ln}]")
    print(f"  genes scored:       {len(df):,}")
    print(f"  padj_treatment<0.05: {(df['padj_treatment'] < 0.05).sum()}")
    print(f"  padj_treatment<0.01: {(df['padj_treatment'] < 0.01).sum()}")
    print(f"  |delta_psi| >= 0.10: {(df['delta_psi_treatment'].abs() >= 0.10).sum()}")
    print(f"  composite_score_abs distribution percentiles:")
    for q in [0.5, 0.9, 0.95, 0.99, 1.0]:
        print(f"    p{int(q*100):3d}: {df['composite_score_abs'].quantile(q):.4f}")